In [29]:
from sklearn import svm
import pandas as pd
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.metrics import make_scorer, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [30]:
def fusionner_dataframes(liste_df):
    # Fusionner les DataFrames de la liste en utilisant la colonne 'a'
    merged_df = liste_df[0]  # Initialiser avec le premier DataFrame de la list
    for df in liste_df[1:]:
        merged_df = pd.merge(merged_df, df, on=['gameId', 'HOME_WON', 'ELO', 'ELO_PROB'])
    return merged_df

In [31]:
# Charger votre jeu de données

#df_l50 = df_l50[['gameId', 'HOME_WON', 'ELO', 'ELO_PROB', 'NB_WIN_L50']]

#liste_df = [df_l10, df_l50]
#data = pd.merge(df_l10, df_l50, on=['gameId', 'HOME_WON', 'ELO', 'ELO_PROB']) 
data = pd.read_csv('../dataset/final_dataset_diff_f_L10.csv', parse_dates=['GAME_DATE'])
data = data.round(2)
#condition = (data['GAME_DATE'] > pd.to_datetime('2013-09-01')) & (data['GAME_DATE'] < pd.to_datetime('2014-09-01'))
condition = (data['GAME_DATE'] > pd.to_datetime('2023-09-01')) & (data['GAME_DATE'] < pd.to_datetime('2024-09-01'))
data_test = data[condition]
data_test = data_test.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])
data_train = data[~condition]
data_train = data_train.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])

#data.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'], inplace=True)


#data_test

In [32]:
"""encoder = OneHotEncoder(sparse=False)
encoded_data = encoder.fit_transform(data['A_teamId'].values.reshape(-1, 1))
encoded_df = pd.DataFrame(encoded_data, columns=['team_' + str(i) for i in range(encoded_data.shape[1])])
data = pd.concat([data, encoded_df], axis=1)
encoder = OneHotEncoder(sparse=False)
encoded_data = encoder.fit_transform(data['H_teamId'].values.reshape(-1, 1))
encoded_df = pd.DataFrame(encoded_data, columns=['team_' + str(i) for i in range(encoded_data.shape[1])])
data = pd.concat([data, encoded_df], axis=1)"""
data

,gameId,GAME_DATE,HOME_WON,H_teamId,A_teamId,ELO,ELO_PROB,NB_WIN_L10,PCT_3PT_L10,PCT_LANCER_FRANC_L10,...,estimatedNetRating_L10,estimatedOffensiveRating_L10,estimatedPace_L10,estimatedTeamTurnoverPercentage_L10,netRating_L10,offensiveRating_L10,pacePer40_L10,pace_L10,trueShootingPercentage_L10,turnoverRatio_L10
0,21300018,2013-10-31,1,1610612741,1610612752,-65.52,0.10,-0.1,0.04,-0.10,...,-24.10,-1.40,4.06,-4.50,-18.50,1.20,3.75,4.50,-0.06,-4.00
1,21300019,2013-10-31,1,1610612746,1610612744,0.52,0.28,-0.1,-0.18,-0.22,...,-36.80,-16.20,-2.46,1.74,-40.80,-13.90,-5.00,-6.00,-0.10,2.00
2,21300020,2013-11-01,1,1610612766,1610612739,-28.04,0.20,-0.1,0.07,-0.03,...,-23.00,-13.50,-4.58,-6.00,-19.60,-15.20,-1.25,-1.50,-0.08,-6.40
3,21300021,2013-11-01,1,1610612753,1610612740,-112.60,-0.04,0.0,-0.03,-0.20,...,-4.45,-2.05,6.75,1.31,-2.60,-0.60,4.95,5.94,0.01,1.75
4,21300022,2013-11-01,0,1610612764,1610612755,-16.70,0.24,-0.1,0.02,-0.08,...,-13.70,-7.20,-3.84,-0.21,-14.10,-7.70,-2.91,-3.50,-0.07,-0.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11894,22301226,2023-12-08,0,1610612757,1610612742,-131.40,-0.09,-0.1,0.03,0.08,...,-6.61,-9.34,-1.92,5.22,-6.22,-8.72,-2.12,-2.54,-0.02,5.50
11895,22301227,2023-12-08,1,1610612738,1610612752,70.75,0.46,0.0,-0.03,-0.01,...,0.35,-3.46,1.57,1.43,-0.36,-4.31,1.57,1.88,0.01,1.36
11896,22301228,2023-12-08,0,1610612756,1610612758,25.74,0.35,0.1,0.02,0.09,...,4.31,1.14,-4.72,1.29,4.54,2.96,-5.12,-6.15,0.01,1.50
11897,22301229,2023-12-07,0,1610612749,1610612754,99.08,0.28,0.3,0.01,-0.01,...,8.28,1.09,-3.27,0.98,6.22,-1.94,-1.42,-1.70,-0.00,0.73


In [33]:
#data_train

In [34]:
data

,gameId,GAME_DATE,HOME_WON,H_teamId,A_teamId,ELO,ELO_PROB,NB_WIN_L10,PCT_3PT_L10,PCT_LANCER_FRANC_L10,...,estimatedNetRating_L10,estimatedOffensiveRating_L10,estimatedPace_L10,estimatedTeamTurnoverPercentage_L10,netRating_L10,offensiveRating_L10,pacePer40_L10,pace_L10,trueShootingPercentage_L10,turnoverRatio_L10
0,21300018,2013-10-31,1,1610612741,1610612752,-65.52,0.10,-0.1,0.04,-0.10,...,-24.10,-1.40,4.06,-4.50,-18.50,1.20,3.75,4.50,-0.06,-4.00
1,21300019,2013-10-31,1,1610612746,1610612744,0.52,0.28,-0.1,-0.18,-0.22,...,-36.80,-16.20,-2.46,1.74,-40.80,-13.90,-5.00,-6.00,-0.10,2.00
2,21300020,2013-11-01,1,1610612766,1610612739,-28.04,0.20,-0.1,0.07,-0.03,...,-23.00,-13.50,-4.58,-6.00,-19.60,-15.20,-1.25,-1.50,-0.08,-6.40
3,21300021,2013-11-01,1,1610612753,1610612740,-112.60,-0.04,0.0,-0.03,-0.20,...,-4.45,-2.05,6.75,1.31,-2.60,-0.60,4.95,5.94,0.01,1.75
4,21300022,2013-11-01,0,1610612764,1610612755,-16.70,0.24,-0.1,0.02,-0.08,...,-13.70,-7.20,-3.84,-0.21,-14.10,-7.70,-2.91,-3.50,-0.07,-0.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11894,22301226,2023-12-08,0,1610612757,1610612742,-131.40,-0.09,-0.1,0.03,0.08,...,-6.61,-9.34,-1.92,5.22,-6.22,-8.72,-2.12,-2.54,-0.02,5.50
11895,22301227,2023-12-08,1,1610612738,1610612752,70.75,0.46,0.0,-0.03,-0.01,...,0.35,-3.46,1.57,1.43,-0.36,-4.31,1.57,1.88,0.01,1.36
11896,22301228,2023-12-08,0,1610612756,1610612758,25.74,0.35,0.1,0.02,0.09,...,4.31,1.14,-4.72,1.29,4.54,2.96,-5.12,-6.15,0.01,1.50
11897,22301229,2023-12-07,0,1610612749,1610612754,99.08,0.28,0.3,0.01,-0.01,...,8.28,1.09,-3.27,0.98,6.22,-1.94,-1.42,-1.70,-0.00,0.73


In [35]:
# Séparer les fonctionnalités (X) de la cible (y)
#X = data.drop('HOME_WON', axis=1)  # Fonctionnalités
#y = data['HOME_WON']  # Cible

X_train = data_train.drop('HOME_WON', axis=1)  # Fonctionnalités
y_train = data_train['HOME_WON']  # Cible

X_test = data_test.drop('HOME_WON', axis=1)  # Fonctionnalités
y_test = data_test['HOME_WON']

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#kernel_matrix = np.dot(X, X.T)
#kernel_matrix_test = np.dot(X_test, X_train.T)
#X = scaler.fit_transform(X)

In [36]:
# Définir les métriques de performance à calculer
scoring = {'accuracy': make_scorer(accuracy_score), 'f1': make_scorer(f1_score)}

In [37]:
#svm_classifier = svm.SVC(kernel='rbf', C=0.05, gamma=0.45)
svm_classifier = svm.SVC(kernel='sigmoid', gamma =0.0145, coef0=0.01)

In [38]:
svm_classifier.fit(X_train, y_train)
#svm_classifier.fit(kernel_matrix_train, y_train)

SVC(coef0=0.01, gamma=0.0145, kernel='sigmoid')

In [39]:
y_pred = svm_classifier.predict(X_test) 
#y_pred = svm_classifier.predict(kernel_matrix_test)
#accuracy = accuracy_score(y_test, y_pred)
#print("Accuracy du modèle SVM: {:.2f}%".format(accuracy * 100))

In [40]:
accuracy = accuracy_score(y_test, y_pred)  # y_test sont les étiquettes de classe réelles des données de test
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("F1-score:", f1)

Accuracy: 0.6425339366515838
F1-score: 0.6982429335370511


In [41]:
# Afficher les scores de performance pour chaque fold
#print("Scores de validation croisée - Accuracy : ", cv_results['test_accuracy'])
#print("Scores de validation croisée - F1 : ", cv_results['test_f1'])

# Calculer la moyenne des scores de performance
#mean_accuracy = cv_results['test_accuracy'].mean()
#mean_f1 = cv_results['test_f1'].mean()
#print("Accuracy moyenne avec validation croisée : {:.2f}%".format(mean_accuracy * 100))
#print("F1 moyenne avec validation croisée : {:.2f}".format(mean_f1))